In [45]:
import ee
import geemap
import geopandas as gpd
import pprint as pp

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

date = '2020-05-29'
date_plus1d = '2020-05-30'
roi_name = 'YKF_sub3'
resamp_method = 'bilinear'


image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}'

idx = 0

def convert_gpd_geom_to_ee(geom, est_utm):
    """
    Takes a geopandas geom object and coverts it to an Earth Engine polygon
    """
    if est_utm is None:
        out_crs = 'EPSG:4326'
    else:
        out_crs = est_utm

    coords = list(geom.exterior.coords)
    coords_list = [[x, y] for x, y in coords]
    return ee.Geometry.Polygon(coords_list, proj=out_crs)

polygon = convert_gpd_geom_to_ee(best_image_dates.geometry[idx], None)

In [46]:
def rescale_s2(img):
    rescaled_bands = img.divide(10_000)
    return rescaled_bands
    
def rescale_ls8(img):
    rescaled_bands = img.multiply(0.0000275).add(-0.2)
    return rescaled_bands

In [47]:
# 1. Reference Landsat image and its projection
ls_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterDate(date, date_plus1d) \
    .filterBounds(polygon) \
    .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5'])

ls_col = ls_col.map(rescale_ls8)


# 2. Sentinel-2 ImageCollection mosaic
s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterDate(date, date_plus1d) \
    .filterBounds(polygon) \
    .select(['B2', 'B3', 'B4', 'B8']) 

s2_col = s2_col.map(rescale_s2)


In [50]:
def repoject_collections(
        s2_col: ee.ImageCollection,
        ls8_col: ee.ImageCollection,
        est_utm: str, 
        resamp_method: str,
        resamp_res: int
    ):

    """
    Reprojects the Sentinel-2 and Landsat 8 image collections to the same UTM zone and resolution
    """
    ls8_fist_img = ls8_col.first()
    ls_proj_orig = ls8_fist_img.projection().getInfo() 

    if resamp_res != 30:
        print(f"sup dumbass")

    ls8_utm_col = ls8_col.map(lambda img: img.reproject(
        crs=est_utm,
        crsTransform=ls_proj_orig['transform'],
    ).resample(resamp_method))

    ls_utm_proj = ls8_utm_col.first().projection().getInfo()

    s2_utm_col = s2_col.map(lambda img: img.reproject(
        crs=ls_utm_proj['crs'],
        crsTransform=ls_utm_proj['transform'],
    ).resample(resamp_method))


    return (s2_utm_col, ls8_utm_col)

In [51]:
utm_cols = repoject_collections(s2_col, ls_col, est_utm, resamp_method, 30)

In [52]:
s2_mosaic = utm_cols[0].mosaic()
ls_mosaic = utm_cols[1].mosaic()

In [ ]:


write_ls = ee.batch.Export.image.toDrive(
    image=ls_utm,
    description=f'ls_image{idx}_{resamp_method}',
    fileNamePrefix=f'ls_image{idx}_{resamp_method}',
    folder='test2',
    crs=ls_utm_proj.getInfo()['crs'],
    crsTransform=ls_utm_proj.getInfo()['transform'],
    region=polygon,
    maxPixels=1e13
)

write_s2 = ee.batch.Export.image.toDrive(
    image=s2_utm,
    description=f's2_image{idx}_{resamp_method}',
    fileNamePrefix=f's2_image{idx}_{resamp_method}',
    folder='test2',
    crs=ls_utm_proj.getInfo()['crs'],
    crsTransform=ls_utm_proj.getInfo()['transform'],
    region=polygon,
    maxPixels=1e13
)

write_ls.start()
write_s2.start()


In [40]:
ls_utm = ls_utm.clip(polygon)
s2_utm = s2_utm.clip(polygon)
pp.pp(ls_utm.projection().getInfo())
pp.pp(s2_utm.projection().getInfo())


{'type': 'Projection',
 'crs': 'EPSG:32607',
 'transform': [30, 0, 254385, 0, -30, 7556115]}
{'type': 'Projection',
 'crs': 'EPSG:32607',
 'transform': [30, 0, 254385, 0, -30, 7556115]}


In [39]:
ls_og_proj = ls.projection().getInfo()

print(f'LandSat orginal crs: {ls_og_proj["crs"]} converting to {est_utm}')
ls_utm = ls.reproject(crs=est_utm, crsTransform=ls_og_proj['transform']).resample(resamp_method)
ls_utm_proj = ls_utm.projection().getInfo()

s2_utm = s2.reproject(crs=ls_utm_proj['crs'], crsTransform=ls_utm_proj['transform']).resample(resamp_method)



pp.pp(ls.projection().getInfo())
pp.pp(s2.projection().getInfo())
pp.pp(ls_utm.projection().getInfo())
pp.pp(s2_utm.projection().getInfo())

LandSat orginal crs: EPSG:32607 converting to EPSG:32607
{'type': 'Projection',
 'crs': 'EPSG:32607',
 'transform': [30, 0, 254385, 0, -30, 7556115]}
{'type': 'Projection',
 'crs': 'EPSG:32606',
 'transform': [10, 0, 600000, 0, -10, 7400040]}
{'type': 'Projection',
 'crs': 'EPSG:32607',
 'transform': [30, 0, 254385, 0, -30, 7556115]}
{'type': 'Projection',
 'crs': 'EPSG:32607',
 'transform': [30, 0, 254385, 0, -30, 7556115]}


In [12]:
Map = geemap.Map(center=[0, 0], zoom=2)

# Visualization parameters for Landsat 8
# Adjust min/max to match your rescaling
vis_params_ls = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # Red, Green, Blue
    'min': 0.0,
    'max': 0.3
}

# Visualization parameters for Sentinel-2
vis_params_s2 = {
    'bands': ['B4', 'B3', 'B2'],          # Red, Green, Blue
    'min': 0.0,
    'max': 0.3
}

# Add the layers to the map
Map.addLayer(ls, vis_params_ls, "Landsat 8")
Map.addLayer(s2, vis_params_s2, "Sentinel-2")
Map.addLayer(ls_utm, vis_params_ls, "Landsat 8 (Reprojected UTM)")
Map.addLayer(s2_utm, vis_params_s2, "Sentinel-2 (Reprojected UTM)")


# Display the map
Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…